In [1]:
import numpy as np
import pandas as pd
import h5py

from SushiLife import *

In [2]:
f = h5py.File("../../../../data/stock_info.hdf5", "r")
array_stock, axis_stock = load_data(f, "stock", chunks=5)
array_value, axis_value = load_data(f, "value", chunks=5)

data_stock = DataAsset(array_stock, axis_stock, chunks=5)
data_value = DataAsset(array_value, axis_value, chunks=5)

# with pandas dataframe

In [3]:
updater = Updater(pd.Timestamp(2003, 1, 1), data_stock.dates)

# 거래소 생성
exchange_stock = Exchange()
exchange_stock.set_DataAsset(data_stock)  # 주가 데이터를 거래소에 등록한다.

# 주식 계좌 생성
stock_account = StockAccount(exchange_stock, 출력=False)

# 거래 에이전트 생성 및 주식 계좌 등록
agent = Agent(1e8, 출력=True)
agent.set_account("stock", stock_account)  # 각 자산을 보관할 지갑을 생성한다.

# 날짜가 변할시 업데이트 요청
updater.set_data(data_stock)
updater.set_data(data_value)

updater.set_exchange(exchange_stock)

updater.set_agent(agent)

updater.initialization()

# 백테스트
columns = ["상장시가총액(원)", "지배주주순이익(원)(직전4분기)", "지배주주지분(원)",
           "현금흐름(원)(직전4분기)", "매출액(원)(직전4분기)"]

#  백테스트 시작
while updater._date != updater._list_date[-1]:
    fin_stat = data_value.get_info(updater._date, num=1,
                                   fields=columns)

    df = pd.DataFrame(fin_stat, index=data_value.codes, columns=columns)
    df = df[~np.isnan(df["상장시가총액(원)"])]  # 상장종목 고려
    df = df.sort_values(by=['상장시가총액(원)']).iloc[:int(len(df.index) * 0.3)]  # 소형주

    # 종목선정
    df["PER"] = df["상장시가총액(원)"] / df["지배주주순이익(원)(직전4분기)"]
    df["PBR"] = df["상장시가총액(원)"] / df["지배주주지분(원)"]
    df["PCR"] = df["상장시가총액(원)"] / df["현금흐름(원)(직전4분기)"]
    df["PSR"] = df["상장시가총액(원)"] / df["매출액(원)(직전4분기)"]

    df = df[df["PER"] > 0]
    df = df[df["PBR"] > 0]
    df = df[df["PCR"] > 0]
    df = df[df["PSR"] > 0]

    df["Rank"] = (df["PER"].rank() + df["PBR"].rank() + df["PCR"].rank() + df["PSR"].rank()).rank()

    df = df[df["Rank"] < 51]

    updater.update()

    # 매도
    매도종목 = agent.accounts["stock"].keys()
    매수종목 = np.sort(df.index)
    현재가 = data_stock.get_info(updater._date, codes=매도종목, fields=["현재가"]).reshape(-1)

    i = 0
    for 종목코드 in 매도종목:
        매도수량 = agent.accounts["stock"][종목코드]["보유수량"]
        agent.sell("stock", 종목코드, 현재가[i], 매도수량, 주문종류="조건부지정가")
        i += 1

    # 매수
    현재가 = data_stock.get_info(updater._date, codes=매수종목, fields=["현재가"]).reshape(-1)
    i = 0
    for 종목코드 in 매수종목:
        if not np.isnan(현재가[i]):
            매수수량 = int(agent.total_balance / 50 / 현재가[i])
            agent.buy("stock", 종목코드, 현재가[i], 매수수량, 주문종류="조건부지정가")
        i += 1

    for i in range(20):
        updater.update()
        if updater._date == updater._list_date[-1]:
            break




 당일수익률(%) :  0.0 
 누적수익률(%) :  0.0 
 CAGR(%) 0.0 
 MDD :  0.0 
 DD :  0.0 
 총자산(원) :  100000000.0 
 --------------------------------------------------
 장시작 :  2003-01-02 00:00:00


 당일수익률(%) :  0.0 
 누적수익률(%) :  0.0 
 CAGR(%) 0.0 
 MDD :  0.0 
 DD :  0.0 
 총자산(원) :  100000000.0 
 --------------------------------------------------
 장시작 :  2003-01-03 00:00:00


 당일수익률(%) :  0.07828 
 누적수익률(%) :  0.07828000000000834 
 CAGR(%) 5.878502229619298 
 MDD :  0.0 
 DD :  0.0 
 총자산(원) :  100078280.0 
 --------------------------------------------------
 장시작 :  2003-01-06 00:00:00


 당일수익률(%) :  -0.45615791958055235 
 누적수익률(%) :  -0.37823500000000454 
 CAGR(%) -20.588675721966666 
 MDD :  -0.4561579195805652 
 DD :  -0.4561579195805652 
 총자산(원) :  99621765.0 
 --------------------------------------------------
 장시작 :  2003-01-07 00:00:00


 당일수익률(%) :  0.0473992806692393 
 누적수익률(%) :  -0.33101500000000117 
 CAGR(%) -15.876731239865727 
 MDD :  -0.4561579195805652 
 DD :  -0.40897485448391946 
 총자



 당일수익률(%) :  2.3514345268284806 
 누적수익률(%) :  -14.997193455000025 
 CAGR(%) -53.24975742169372 
 MDD :  -18.715562912352244 
 DD :  -15.063681605039605 
 총자산(원) :  85002806.54499997 
 --------------------------------------------------
 장시작 :  2003-03-20 00:00:00


 당일수익률(%) :  0.8289608645180061 
 누적수익률(%) :  -14.292553455000023 
 CAGR(%) -50.962448901467816 
 MDD :  -18.715562912352244 
 DD :  -14.359592765782974 
 총자산(원) :  85707446.54499997 
 --------------------------------------------------
 장시작 :  2003-03-21 00:00:00


 당일수익률(%) :  0.5010591463304457 
 누적수익률(%) :  -13.863108455000027 
 CAGR(%) -48.53489290030769 
 MDD :  -18.715562912352244 
 DD :  -13.930483672381294 
 총자산(원) :  86136891.54499997 
 --------------------------------------------------
 장시작 :  2003-03-24 00:00:00


 당일수익률(%) :  -2.9405286800682413 
 누적수익률(%) :  -16.395988455000033 
 CAGR(%) -54.50258551586549 
 MDD :  -18.715562912352244 
 DD :  -16.46138248479095 
 총자산(원) :  83604011.54499997 
 ------------------



 당일수익률(%) :  -0.2544941552829851 
 누적수익률(%) :  -2.8350638290000396 
 CAGR(%) -6.057296713573901 
 MDD :  -18.715562912352244 
 DD :  -2.911065047281036 
 총자산(원) :  97164936.17099996 
 --------------------------------------------------
 장시작 :  2003-06-18 00:00:00


 당일수익률(%) :  -0.2527557879189955 
 누적수익률(%) :  -3.0806538290000463 
 CAGR(%) -6.534820599157543 
 MDD :  -18.715562912352244 
 DD :  -3.156462949802949 
 총자산(원) :  96919346.17099996 
 --------------------------------------------------
 장시작 :  2003-06-19 00:00:00


 당일수익률(%) :  0.08505525806432448 
 누적수익률(%) :  -2.9982188290000433 
 CAGR(%) -6.326821369542957 
 MDD :  -18.715562912352244 
 DD :  -3.0740924294462806 
 총자산(원) :  97001781.17099996 
 --------------------------------------------------
 장시작 :  2003-06-20 00:00:00


 당일수익률(%) :  0.09780232780752557 
 누적수익률(%) :  -2.903348829000041 
 CAGR(%) -6.026978779956982 
 MDD :  -18.715562912352244 
 DD :  -2.9792966355937063 
 총자산(원) :  97096651.17099996 
 ------------------

 MDD :  -18.715562912352244 
 DD :  -1.218547044088583 
 총자산(원) :  100622812.52249996 
 --------------------------------------------------
 장시작 :  2003-09-02 00:00:00


 당일수익률(%) :  -0.4940877595613126 
 누적수익률(%) :  0.12564752249997202 
 CAGR(%) 0.18724675351529108 
 MDD :  -18.715562912352244 
 DD :  -1.7066141118605467 
 총자산(원) :  100125647.52249996 
 --------------------------------------------------
 장시작 :  2003-09-03 00:00:00


 당일수익률(%) :  0.48622906522586895 
 누적수익률(%) :  0.6124875224999604 
 CAGR(%) 0.9101169973522616 
 MDD :  -18.715562912352244 
 DD :  -1.2286831004778014 
 총자산(원) :  100612487.52249996 
 --------------------------------------------------
 장시작 :  2003-09-04 00:00:00


 당일수익률(%) :  -0.5451374734943629 
 누적수익률(%) :  0.06401114999996516 
 CAGR(%) 0.09460583702669556 
 MDD :  -18.715562912352244 
 DD :  -1.7671225619609752 
 총자산(원) :  100064011.14999998 
 --------------------------------------------------
 장시작 :  2003-09-05 00:00:00


 당일수익률(%) :  -0.6361228104736



 당일수익률(%) :  0.3533368586799444 
 누적수익률(%) :  1.9461389715 
 CAGR(%) 2.201964352882624 
 MDD :  -18.715562912352244 
 DD :  -1.182639969780291 
 총자산(원) :  101946138.97150001 
 --------------------------------------------------
 장시작 :  2003-11-20 00:00:00


 당일수익률(%) :  0.24690488775682604 
 누적수익률(%) :  2.197848971500016 
 CAGR(%) 2.4793931482064213 
 MDD :  -18.715562912352244 
 DD :  -0.9386550779134029 
 총자산(원) :  102197848.97150001 
 --------------------------------------------------
 장시작 :  2003-11-21 00:00:00


 당일수익률(%) :  -1.3894910844904425 
 누적수익률(%) :  0.7778189714999995 
 CAGR(%) 0.868599205697107 
 MDD :  -18.715562912352244 
 DD :  -2.3151036337821376 
 총자산(원) :  100777818.97150001 
 --------------------------------------------------
 장시작 :  2003-11-24 00:00:00


 당일수익률(%) :  1.2945209703029377 
 누적수익률(%) :  2.082408971500005 
 CAGR(%) 2.3200200173554064 
 MDD :  -18.715562912352244 
 DD :  -1.0505521655027497 
 총자산(원) :  102082408.97150001 
 ----------------------------

...



 당일수익률(%) :  0.5054179269538588 
 누적수익률(%) :  11044.217375497024 
 CAGR(%) 36.92097140730459 
 MDD :  -52.57681552529717 
 DD :  -15.613455665984516 
 총자산(원) :  11144217375.497025 
 --------------------------------------------------
 장시작 :  2017-12-28 00:00:00


 당일수익률(%) :  0.21675438647770165 
 누적수익률(%) :  11068.372955497025 
 CAGR(%) 36.901462276117456 
 MDD :  -52.57681552529717 
 DD :  -15.430544129543582 
 총자산(원) :  11168372955.497025 
 --------------------------------------------------
 장시작 :  2018-01-02 00:00:00


 당일수익률(%) :  0.2820943939241678 
 누적수익률(%) :  11099.878309497026 
 CAGR(%) 36.91929993650363 
 MDD :  -52.57681552529717 
 DD :  -15.191978435560847 
 총자산(원) :  11199878309.497025 
 --------------------------------------------------
 장시작 :  2018-01-03 00:00:00


 당일수익률(%) :  -0.09217623365823538 
 누적수익률(%) :  11089.554683497025 
 CAGR(%) 36.903045919908116 
 MDD :  -52.57681552529717 
 DD :  -15.27015127567902 
 총자산(원) :  11189554683.497025 
 -----------------------



 당일수익률(%) :  -0.37061348389163457 
 누적수익률(%) :  12058.349737090726 
 CAGR(%) 37.05389757930075 
 MDD :  -52.57681552529717 
 DD :  -7.934215158674332 
 총자산(원) :  12158349737.090727 
 --------------------------------------------------
 장시작 :  2018-03-22 00:00:00


 당일수익률(%) :  -2.5038571975874424 
 누적수익률(%) :  11753.922022090726 
 CAGR(%) 36.81818430940875 
 MDD :  -52.57681552529717 
 DD :  -10.23941093893923 
 총자산(원) :  11853922022.090727 
 --------------------------------------------------
 장시작 :  2018-03-23 00:00:00


 당일수익률(%) :  0.9794910054547628 
 누적수익률(%) :  11870.030122090726 
 CAGR(%) 36.882570119490055 
 MDD :  -52.57681552529717 
 DD :  -9.360214042642934 
 총자산(원) :  11970030122.090727 
 --------------------------------------------------
 장시작 :  2018-03-26 00:00:00


 당일수익률(%) :  0.48364448050267994 
 누적수익률(%) :  11927.922512090727 
 CAGR(%) 36.91817538562023 
 MDD :  -52.57681552529717 
 DD :  -8.921839720720726 
 총자산(원) :  12027922512.090727 
 --------------------------



 당일수익률(%) :  -2.0017610070645864 
 누적수익률(%) :  13147.178466045727 
 CAGR(%) 37.17121553180387 
 MDD :  -52.57681552529717 
 DD :  -6.7201515709228055 
 총자산(원) :  13247178466.045727 
 --------------------------------------------------
 장시작 :  2018-06-14 00:00:00


 당일수익률(%) :  0.20891856383558793 
 누적수익률(%) :  13174.854281045727 
 CAGR(%) 37.18204814974535 
 MDD :  -52.57681552529717 
 DD :  -6.525272651236761 
 총자산(원) :  13274854281.045727 
 --------------------------------------------------
 장시작 :  2018-06-15 00:00:00


 당일수익률(%) :  -0.8823313500867538 
 누적수익률(%) :  13057.726080045728 
 CAGR(%) 37.08046326847199 
 MDD :  -52.57681552529717 
 DD :  -7.350029475043011 
 총자산(원) :  13157726080.045727 
 --------------------------------------------------
 장시작 :  2018-06-18 00:00:00


 당일수익률(%) :  -2.3099259488379906 
 누적수익률(%) :  12753.792351045726 
 CAGR(%) 36.865944765493765 
 MDD :  -52.57681552529717 
 DD :  -9.490175185789758 
 총자산(원) :  12853792351.045727 
 -------------------------



 당일수익률(%) :  0.29049613871074914 
 누적수익률(%) :  12628.117865969925 
 CAGR(%) 36.18206794265406 
 MDD :  -52.57681552529717 
 DD :  -10.375110566506514 
 총자산(원) :  12728117865.969925 
 --------------------------------------------------
 장시작 :  2018-09-07 00:00:00


 당일수익률(%) :  2.487553842084669 
 누적수익률(%) :  12944.736650969924 
 CAGR(%) 36.37329926336894 
 MDD :  -52.57681552529717 
 DD :  -8.14564318593952 
 총자산(원) :  13044736650.969925 
 --------------------------------------------------
 장시작 :  2018-09-10 00:00:00


 당일수익률(%) :  0.2436105341654226 
 누적수익률(%) :  12976.515003605826 
 CAGR(%) 36.38704842515832 
 MDD :  -52.57681552529717 
 DD :  -7.921876296650565 
 총자산(원) :  13076515003.605825 
 --------------------------------------------------
 장시작 :  2018-09-11 00:00:00


 당일수익률(%) :  -0.4190516355840201 
 누적수익률(%) :  12921.717653605825 
 CAGR(%) 36.343208955449626 
 MDD :  -52.57681552529717 
 DD :  -8.307731180044534 
 총자산(원) :  13021717653.605825 
 -----------------------------



 당일수익률(%) :  -0.9938754937572674 
 누적수익률(%) :  11046.749920935525 
 CAGR(%) 34.49873312646934 
 MDD :  -52.57681552529717 
 DD :  -21.510294002096007 
 총자산(원) :  11146749920.935526 
 --------------------------------------------------
 장시작 :  2018-11-23 00:00:00


 당일수익률(%) :  0.6738877837289419 
 누적수익률(%) :  11121.866506935525 
 CAGR(%) 34.53491662120012 
 MDD :  -52.57681552529717 
 DD :  -20.981361461891364 
 총자산(원) :  11221866506.935526 
 --------------------------------------------------
 장시작 :  2018-11-26 00:00:00


 당일수익률(%) :  0.8563730992576504 
 누적수익률(%) :  11217.967552935526 
 CAGR(%) 34.60014562018785 
 MDD :  -52.57681552529717 
 DD :  -20.304667098051354 
 총자산(원) :  11317967552.935526 
 --------------------------------------------------
 장시작 :  2018-11-27 00:00:00


 당일수익률(%) :  0.36492959364676214 
 누적수익률(%) :  11259.270165935526 
 CAGR(%) 34.62406603337094 
 MDD :  -52.57681552529717 
 DD :  -20.01383524353684 
 총자산(원) :  11359270165.935526 
 --------------------------

In [4]:
agent.stat("templete-pandas")

TypeError: stat() missing 1 required positional argument: 'Strategy_name'

# Pure numpy

dataframe을 사용하지 않고 numpy만 사용할 경우 3배까지 속도향상 가능

In [6]:
%%time
updater = Updater(pd.Timestamp(2002, 6, 15), data_stock.dates)

# 거래소 생성
exchange_stock = Exchange()
exchange_stock.set_DataAsset(data_stock)

# 주식 계좌 생성
stock_account = StockAccount(exchange_stock, 출력=False)

# 거래 에이전트 생성 및 주식 계좌 등록
agent = Agent(1e8, 출력=True)
agent.set_account("stock", stock_account)

# 날짜가 변할시 업데이트 요청
updater.set_data(data_stock)
updater.set_data(data_value)

updater.set_exchange(exchange_stock)

updater.set_agent(agent)

updater.initialization()

# 백테스트

columns = ["상장시가총액(원)", "지배주주순이익(원)(직전4분기)", "지배주주지분(원)",
   "현금흐름(원)(직전4분기)", "매출액(원)(직전4분기)"]

while updater._date != updater._list_date[-1]:
    fin_stat = data_value.get_info(updater._date, num=1,
                             fields=columns)
    
    상장시가총액 = fin_stat[:, 0]
    
    # 상장종목 고려
    상장종목 = ~np.isnan(상장시가총액)
    
    종목코드 = np.array(data_value.codes)[상장종목]
    상장시가총액 = 상장시가총액[상장종목]
    values = 상장시가총액.reshape(-1, 1) / fin_stat[상장종목, 1:]
    
    # 소형주
    시가총액순위 = 상장시가총액.argsort().argsort()
    시가총액조건 = 시가총액순위 < len(시가총액순위) * 0.3 # 상장종목 & 소형주
    
    values = values[시가총액조건, :]
    종목코드 = 종목코드[시가총액조건]
    
    # 양수
    cond_positive = (values > 0).all(axis=1)
    values = values[cond_positive, :]
    종목코드 = 종목코드[cond_positive]

    rank_each = values.argsort(axis=0).argsort(axis=0)
    sum_rank = np.sum(rank_each, axis=1)
    rank = sum_rank.argsort(axis=0).argsort(axis=0)

    cond_rank = rank < 50
    매수종목 = 종목코드[cond_rank]
    updater.update()

    # 매도
    매도종목 = agent.accounts["stock"].keys()
    현재가 = data_stock.get_info(updater._date, codes=매도종목, fields=["현재가"]).reshape(-1)
    
    i = 0
    for 종목코드 in 매도종목:
        매도수량 = agent.accounts["stock"][종목코드]["보유수량"]
        agent.sell("stock", 종목코드, 현재가[i], 매도수량)
        i += 1

    # 매수
    현재가 = data_stock.get_info(updater._date, codes=매수종목, fields=["현재가"]).reshape(-1)
    i = 0
    for 종목코드 in 매수종목:
        if not np.isnan(현재가[i]):
            매수수량 = int(agent.total_balance / 50 / 현재가[i])
            agent.buy("stock", 종목코드, 현재가[i], 매수수량, 주문종류="조건부지정가")
        i += 1

    for i in range(20):
        if updater._date == updater._list_date[-1]:
            break
        updater.update()

c:\users\wkwek\appdata\local\programs\python\python36\lib\site-packages\ipykernel_launcher.py:50: RuntimeWarning:

invalid value encountered in greater





 당일수익률(%) :  0.0 
 누적수익률(%) :  0.0 
 CAGR(%) 0.0 
 MDD :  0.0 
 DD :  0.0 
 총자산(원) :  100000000.0 
 --------------------------------------------------
 장시작 :  2002-06-17 00:00:00


 당일수익률(%) :  0.0 
 누적수익률(%) :  0.0 
 CAGR(%) 0.0 
 MDD :  0.0 
 DD :  0.0 
 총자산(원) :  100000000.0 
 --------------------------------------------------
 장시작 :  2002-06-18 00:00:00


 당일수익률(%) :  -3.53522 
 누적수익률(%) :  -3.5352199999999945 
 CAGR(%) -98.74627771707088 
 MDD :  -3.5352199999999945 
 DD :  -3.5352199999999945 
 총자산(원) :  96464780.0 
 --------------------------------------------------
 장시작 :  2002-06-19 00:00:00


 당일수익률(%) :  0.2464889258027645 
 누적수익률(%) :  -3.297444999999999 
 CAGR(%) -95.30956814748257 
 MDD :  -3.5352199999999945 
 DD :  -3.297444999999999 
 총자산(원) :  96702555.0 
 --------------------------------------------------
 장시작 :  2002-06-20 00:00:00


 당일수익률(%) :  0.043003000282670914 
 누적수익률(%) :  -3.2558599999999993 
 CAGR(%) -91.07515281255402 
 MDD :  -3.5352199999999945 
 DD :



 당일수익률(%) :  -0.03500261934684552 
 누적수익률(%) :  -10.780776511999967 
 CAGR(%) -55.798414436033305 
 MDD :  -13.2455 
 DD :  -10.780776511999967 
 총자산(원) :  89219223.48800004 
 --------------------------------------------------
 장시작 :  2002-08-06 00:00:00


 당일수익률(%) :  0.917576916716955 
 누적수익률(%) :  -9.962121511999966 
 CAGR(%) -52.12593154552758 
 MDD :  -13.2455 
 DD :  -9.962121511999966 
 총자산(원) :  90037878.48800004 
 --------------------------------------------------
 장시작 :  2002-08-07 00:00:00


 당일수익률(%) :  0.7948154843468215 
 누적수익률(%) :  -9.246486511999963 
 CAGR(%) -48.735785315894084 
 MDD :  -13.2455 
 DD :  -9.246486511999963 
 총자산(원) :  90753513.48800004 
 --------------------------------------------------
 장시작 :  2002-08-08 00:00:00


 당일수익률(%) :  0.5591684338117664 
 누적수익률(%) :  -8.739021511999967 
 CAGR(%) -46.10396259264773 
 MDD :  -13.2455 
 DD :  -8.739021511999967 
 총자산(원) :  91260978.48800004 
 --------------------------------------------------
 장시작 :  2002-08

 누적수익률(%) :  -9.051652969500024 
 CAGR(%) -29.027491442205346 
 MDD :  -13.2455 
 DD :  -9.7005169337491 
 총자산(원) :  90948347.03049998 
 --------------------------------------------------
 장시작 :  2002-09-25 00:00:00


 당일수익률(%) :  1.4030491390591961 
 누적수익률(%) :  -7.775602969500017 
 CAGR(%) -25.14803680512282 
 MDD :  -13.2455 
 DD :  -8.433570814013153 
 총자산(원) :  92224397.03049998 
 --------------------------------------------------
 장시작 :  2002-09-26 00:00:00


 당일수익률(%) :  -0.6212239043542099 
 누적수익률(%) :  -8.348522969500017 
 CAGR(%) -26.57667661243327 
 MDD :  -13.2455 
 DD :  -9.002403360480074 
 총자산(원) :  91651477.03049998 
 --------------------------------------------------
 장시작 :  2002-09-27 00:00:00


 당일수익률(%) :  -3.2446176497629082 
 누적수익률(%) :  -11.322262969500018 
 CAGR(%) -33.88423965100158 
 MDD :  -13.2455 
 DD :  -11.954927441905998 
 총자산(원) :  88677737.03049998 
 --------------------------------------------------
 장시작 :  2002-09-30 00:00:00


 당일수익률(%) :  0.3317123



 당일수익률(%) :  -3.3924256831757664 
 누적수익률(%) :  -13.693729281000044 
 CAGR(%) -24.641197174840723 
 MDD :  -14.610708823958014 
 DD :  -14.309474709990678 
 총자산(원) :  86306270.71899995 
 --------------------------------------------------
 장시작 :  2002-12-23 00:00:00


 당일수익률(%) :  -2.298975478224661 
 누적수익률(%) :  -15.677889281000045 
 CAGR(%) -27.81041665899462 
 MDD :  -16.279478873569893 
 DD :  -16.279478873569893 
 총자산(원) :  84322110.71899995 
 --------------------------------------------------
 장시작 :  2002-12-24 00:00:00


 당일수익률(%) :  0.4572347593205179 
 누적수익률(%) :  -15.292339281000045 
 CAGR(%) -26.938600663480237 
 MDD :  -16.279478873569893 
 DD :  -15.896679550295579 
 총자산(원) :  84707660.71899995 
 --------------------------------------------------
 장시작 :  2002-12-26 00:00:00


 당일수익률(%) :  -2.21536043383982 
 누적수익률(%) :  -17.16891928100005 
 CAGR(%) -29.840677804936654 
 MDD :  -17.75987123508385 
 DD :  -17.75987123508385 
 총자산(원) :  82831080.71899995 
 -------------------

...



 당일수익률(%) :  -0.6618918425286515 
 누적수익률(%) :  8259.260746678898 
 CAGR(%) 32.99177802555355 
 MDD :  -59.57699737787756 
 DD :  -13.90678645541743 
 총자산(원) :  8359260746.678898 
 --------------------------------------------------
 장시작 :  2017-12-20 00:00:00


 당일수익률(%) :  -1.9816242080103579 
 누적수익률(%) :  8093.611612112001 
 CAGR(%) 32.81376124723736 
 MDD :  -59.57699737787756 
 DD :  -15.612830416470933 
 총자산(원) :  8193611612.112001 
 --------------------------------------------------
 장시작 :  2017-12-21 00:00:00


 당일수익률(%) :  0.00761379754780928 
 누적수익률(%) :  8094.235457112002 
 CAGR(%) 32.80776300976365 
 MDD :  -59.57699737787756 
 DD :  -15.606405348222504 
 총자산(원) :  8194235457.112001 
 --------------------------------------------------
 장시작 :  2017-12-22 00:00:00


 당일수익률(%) :  -0.462734030507878 
 누적수익률(%) :  8056.317941112002 
 CAGR(%) 32.74156695018904 
 MDD :  -59.57699737787756 
 DD :  -15.99692323024516 
 총자산(원) :  8156317941.112001 
 ----------------------------------



 당일수익률(%) :  -2.1604558716231894 
 누적수익률(%) :  8585.544989105203 
 CAGR(%) 32.70250892761857 
 MDD :  -59.57699737787756 
 DD :  -10.546338706422324 
 총자산(원) :  8685544989.105204 
 --------------------------------------------------
 장시작 :  2018-03-23 00:00:00


 당일수익률(%) :  1.0844643613975884 
 누적수익률(%) :  8679.736629105204 
 CAGR(%) 32.77365042511606 
 MDD :  -59.57699737787756 
 DD :  -9.576245629728156 
 총자산(원) :  8779736629.105204 
 --------------------------------------------------
 장시작 :  2018-03-26 00:00:00


 당일수익률(%) :  0.5733753428675518 
 누적수익률(%) :  8730.077474105205 
 CAGR(%) 32.81520467375856 
 MDD :  -59.57699737787756 
 DD :  -9.05777811807389 
 총자산(원) :  8830077474.105204 
 --------------------------------------------------
 장시작 :  2018-03-27 00:00:00


 당일수익률(%) :  -0.4869632734038679 
 누적수익률(%) :  8687.078239793203 
 CAGR(%) 32.767618483318174 
 MDD :  -59.57699737787756 
 DD :  -9.500633338656336 
 총자산(원) :  8787078239.793203 
 ------------------------------------

 MDD :  -59.57699737787756 
 DD :  -8.60141010988352 
 총자산(원) :  9424570159.588207 
 --------------------------------------------------
 장시작 :  2018-06-27 00:00:00


 당일수익률(%) :  -1.8503602291355816 
 누적수익률(%) :  9150.181661588207 
 CAGR(%) 32.60156252627748 
 MDD :  -59.57699737787756 
 DD :  -10.292613267200972 
 총자산(원) :  9250181661.588207 
 --------------------------------------------------
 장시작 :  2018-06-28 00:00:00


 당일수익률(%) :  -0.5119489511935617 
 누적수익률(%) :  9102.825453588208 
 CAGR(%) 32.552769242527816 
 MDD :  -59.57699737787756 
 DD :  -10.75186929272268 
 총자산(원) :  9202825453.588207 
 --------------------------------------------------
 장시작 :  2018-06-29 00:00:00


 당일수익률(%) :  -3.411352531842218 
 누적수익률(%) :  8788.884634476206 
 CAGR(%) 32.24743109210932 
 MDD :  -59.57699737787756 
 DD :  -13.796437659227243 
 총자산(원) :  8888884634.476206 
 --------------------------------------------------
 장시작 :  2018-07-02 00:00:00


 당일수익률(%) :  -0.17288976774874734 
 누적수익률(%) :  8

 당일수익률(%) :  0.10716415614600147 
 누적수익률(%) :  8906.349050860304 
 CAGR(%) 32.066953263026555 
 MDD :  -59.57699737787756 
 DD :  -12.657278860685157 
 총자산(원) :  9006349050.860306 
 --------------------------------------------------
 장시작 :  2018-08-17 00:00:00


 당일수익률(%) :  0.16641166043379532 
 누적수익률(%) :  8921.336665860306 
 CAGR(%) 32.06186828563753 
 MDD :  -59.57699737787756 
 DD :  -12.511930388169146 
 총자산(원) :  9021336665.860306 
 --------------------------------------------------
 장시작 :  2018-08-20 00:00:00


 당일수익률(%) :  1.1518243786780435 
 누적수익률(%) :  9025.246620860305 
 CAGR(%) 32.14908987942244 
 MDD :  -59.57699737787756 
 DD :  -11.50422147394527 
 총자산(원) :  9125246620.860306 
 --------------------------------------------------
 장시작 :  2018-08-21 00:00:00


 당일수익률(%) :  0.04808387304261517 
 누적수익률(%) :  9029.634392860307 
 CAGR(%) 32.14678055205267 
 MDD :  -59.57699737787756 
 DD :  -11.461669276150708 
 총자산(원) :  9129634392.860306 
 ----------------------------------



 당일수익률(%) :  0.8995546366481496 
 누적수익률(%) :  8501.50669497191 
 CAGR(%) 31.12533699385256 
 MDD :  -59.57699737787756 
 DD :  -16.583401732012877 
 총자산(원) :  8601506694.97191 
 --------------------------------------------------
 장시작 :  2018-11-19 00:00:00


 당일수익률(%) :  -1.0047630963366032 
 누적수익률(%) :  8415.08192997191 
 CAGR(%) 31.038904548946554 
 MDD :  -59.57699737787756 
 DD :  -17.421540927628968 
 총자산(원) :  8515081929.97191 
 --------------------------------------------------
 장시작 :  2018-11-20 00:00:00


 당일수익률(%) :  0.3160902293289947 
 누적수익률(%) :  8441.99727197191 
 CAGR(%) 31.058153217167785 
 MDD :  -59.57699737787756 
 DD :  -17.16051848697076 
 총자산(원) :  8541997271.97191 
 --------------------------------------------------
 장시작 :  2018-11-21 00:00:00


 당일수익률(%) :  -0.8898252314994411 
 누적수익률(%) :  8365.98842497191 
 CAGR(%) 30.98104591525106 
 MDD :  -59.57699737787756 
 DD :  -17.897645095117 
 총자산(원) :  8465988424.97191 
 -------------------------------------------

In [8]:
agent.stat("templete-numpy")

http://localhost:8097/#


,수익률(%),누적수익률(%),총자산(원),현금자산,현물자산,CAGR(%),일평균수익률(%),MDD,DD,최대수익률(%),날짜
0,0.000000,0.000000,1.000000e+08,1.000000e+08,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,2002-06-17
1,0.000000,0.000000,1.000000e+08,1.000000e+08,1.000000e+08,0.000000,0.000000,0.000000,0.000000,0.000000,2002-06-17
2,0.000000,0.000000,1.000000e+08,6.188470e+06,1.000000e+08,0.000000,0.000000,0.000000,0.000000,0.000000,2002-06-18
3,-3.535220,-3.535220,9.646478e+07,6.188470e+06,9.646478e+07,-98.746278,-1.192572,-3.535220,-3.535220,0.000000,2002-06-19
4,0.246489,-3.297445,9.670256e+07,6.188470e+06,9.670256e+07,-95.309568,-0.834755,-3.535220,-3.297445,0.000000,2002-06-20
5,0.043003,-3.255860,9.674414e+07,6.188470e+06,9.674414e+07,-91.075153,-0.659822,-3.535220,-3.255860,0.000000,2002-06-21
6,-2.440168,-5.616580,9.438342e+07,6.188470e+06,9.438342e+07,-92.844882,-0.719955,-5.616580,-5.616580,0.000000,2002-06-24
7,-0.728772,-6.304420,9.369558e+07,6.188470e+06,9.369558e+07,-92.870607,-0.720935,-6.304420,-6.304420,0.000000,2002-06-25
8,-7.408119,-13.245500,8.675450e+07,6.188470e+06,8.675450e+07,-99.440683,-1.410832,-13.245500,-13.245500,0.000000,2002-06-26
9,2.137434,-11.391180,8.860882e+07,6.188470e+06,8.860882e+07,-98.192036,-1.093422,-13.245500,-11.391180,0.000000,2002-06-27
